In [19]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import warnings

# =========================================
# Sacar warnings
# =========================================

warnings.filterwarnings("ignore")

# =========================================
# Carpeta con excels
# =========================================

carpeta = Path("/Users/ri1965/Proyectos/PAC_v2/raw")
archivos_excel = list(carpeta.glob("*.xlsx"))

# =========================================
# Variables de control
# =========================================

total_archivos = 0
archivos_sin_discharge = []
archivos_con_discharge = []
archivos_sin_columna_discharge = []
archivos_error = []

In [20]:
# =========================================
# Loop con barra de progreso
# =========================================

for archivo in tqdm(archivos_excel, desc="Procesando Excels"):
    try:
        df = pd.read_excel(
            archivo,
            skiprows=6
        )
        total_archivos += 1
        df.columns = df.columns.str.strip().str.lower()

        if "discharge_level" not in df.columns:
            archivos_sin_columna_discharge.append({
                "archivo": archivo.name
            })
            continue

        # Detectar datos reales en discharge
        discharge_con_datos = (
            df["discharge_level"].notna() &
            (df["discharge_level"].astype(str).str.strip() != "")
        )
        n_con_datos = int(discharge_con_datos.sum())
        n_vacios = int((~discharge_con_datos).sum())

        # Caso 1: ninguna descarga registrada
        if n_con_datos == 0:
            archivos_sin_discharge.append({
                "archivo": archivo.name,
                "filas_totales": len(df),
                "filas_discharge_vacias": n_vacios
            })

        # Caso 2: al menos una descarga registrada
        else:
            archivos_con_discharge.append({
                "archivo": archivo.name,
                "filas_totales": len(df),
                "filas_con_discharge": n_con_datos,
                "filas_discharge_vacias": n_vacios
            })

    except Exception as e:
        archivos_error.append({
            "archivo": archivo.name,
            "error": str(e)
        })

Procesando Excels: 100%|██████████████████████| 562/562 [57:52<00:00,  6.18s/it]


In [21]:
# =========================================
# DataFrames
# =========================================

df_sin_discharge = pd.DataFrame(archivos_sin_discharge)
df_con_discharge = pd.DataFrame(archivos_con_discharge)
df_sin_columna = pd.DataFrame(archivos_sin_columna_discharge)
df_errores = pd.DataFrame(archivos_error)

In [22]:
# =========================================
# Resultados
# =========================================

print("\n===================================")
print("RESUMEN")
print("===================================")
print(f"Total archivos encontrados: {len(archivos_excel)}")
print(f"Total archivos analizados: {total_archivos}")
print(f"Archivos SIN discharge: {len(df_sin_discharge)}")
print(f"Archivos CON discharge: {len(df_con_discharge)}")
print(f"Archivos sin columna discharge: {len(df_sin_columna)}")
print(f"Archivos con error: {len(df_errores)}")

print("\n===================================")
print("ARCHIVOS SIN DISCHARGE")
print("===================================")
print(df_sin_discharge)

print("\n===================================")
print("ARCHIVOS CON DISCHARGE")
print("===================================")
print(df_con_discharge)


RESUMEN
Total archivos encontrados: 562
Total archivos analizados: 561
Archivos SIN discharge: 4
Archivos CON discharge: 556
Archivos sin columna discharge: 1
Archivos con error: 1

ARCHIVOS SIN DISCHARGE
           archivo  filas_totales  filas_discharge_vacias
0  exam_12237.xlsx            786                     786
1  exam_11704.xlsx           1730                    1730
2  exam_12229.xlsx           1194                    1194
3  exam_11961.xlsx           2859                    2859

ARCHIVOS CON DISCHARGE
             archivo  filas_totales  filas_con_discharge  \
0    exam_11796.xlsx          34360                   71   
1    exam_12440.xlsx          24705                  267   
2    exam_12505.xlsx          24979                   10   
3    exam_13178.xlsx          25001                  179   
4    exam_12293.xlsx          30420                   24   
..               ...            ...                  ...   
551  exam_12984.xlsx          23899                   19   


In [ ]:
'''# =========================================
# Guardar CSVs para futuros subsets
# =========================================

df_sin_discharge.to_csv(carpeta / "subset_archivos_sin_discharge.csv", index=False)
df_con_discharge.to_csv(carpeta / "subset_archivos_con_discharge.csv", index=False)

if len(df_sin_columna) > 0:
    df_sin_columna.to_csv(carpeta / "archivos_sin_columna_discharge.csv", index=False)

if len(df_errores) > 0:
    df_errores.to_csv(carpeta / "errores_lectura_excel.csv", index=False)

print("\nCSVs guardados:")
print(carpeta / "subset_archivos_sin_discharge.csv")
print(carpeta / "subset_archivos_con_discharge.csv")'''

In [18]:
pd.read_excel(archivos_excel[0], skiprows=6).head()

,somni_epoch,time,spo2,bpm,acceleration_module,discharge_level,battery_level,temperature,sleep_stage,Unnamed: 9,...,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21
0,817003005,10:16:45 PM,0,0,172,,,,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,817003006,10:16:46 PM,0,0,43,,,,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,817003007,10:16:47 PM,0,0,78,,,,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,817003008,10:16:48 PM,0,0,129,,,,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,817003009,10:16:49 PM,0,0,13,,,,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
